# Sloshing tank (SPHERIC test case 10)

Water in a rolling tank, lateral impact: the wall pressure at sensor 1 from a warpSPH run against the SPHERIC experiment, with the same functions the [case page](https://fluids.dev/cases/sloshing-tank/) is built from. The run here stops after the first impact at a coarse resolution (a few minutes on a GPU); the page's runs go to 7.5 s at `nx = 200` and finer. In Colab choose *Runtime > Change runtime type > GPU* first.

In [ ]:
# Setup: installs the latest warpSPH stack from GitHub when it is not already present.
import importlib.util, subprocess, sys
if importlib.util.find_spec('warpSPHCases') is None:
    stack = ['warpSPHCore', 'warpSPHIntegrators', 'warpSPHPlotting', 'warpSPH', 'warpSPHCases']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q']
                          + [f'git+https://github.com/wi-re/{name}' for name in stack])
    subprocess.call('apt-get -qq install -y xvfb > /dev/null', shell=True)
    from pyvirtualdisplay import Display
    Display(visible=0, size=(1280, 800)).start()

## Parameters

`nx` is the tank width divided by the particle spacing (the page uses 200); `tLimit` the simulated time (the first impact is at 2.4 s, the four recorded impacts end at 7.3 s).

In [ ]:
scheme = 'deltaSPH'
nx = 100
tLimit = 2.6
video = True

## Experiment data

The reduced roll history and the repeatability register of the experiment are fetched from the repository (see `PROVENANCE.md` next to them for what was derived from the SPHERIC files).

In [ ]:
from warpSPHCases import quick

FILES = ['reference/spheric-tc10/lateral_water_1kHz.txt',
         'reference/spheric-tc10/Repeatability_Files/Water_4first_peak_lateral_impact_tto_0_85_H93_B1X.txt']
page = quick.fetchPage('free-surface', 'sloshing-tank', FILES)

## Run

In [ ]:
traj = quick.run('sloshingTank', scheme=scheme, nx=nx, video=video,
                 spec={'tLimit': tLimit, 'exportInterval': 0.02},
                 params={'rollDataFile': str((page / FILES[0]).resolve())})

## Sensor pressure against the experiment

Grey: the experiment (1 ms mean and the 1 ms maximum behind it). Black bar: mean and standard deviation of the impact peak over the repeated experiments. Impacts beyond `tLimit` are not reached.

In [ ]:
fig, metrics = quick.sensor(traj, 'tc10', page)
metrics

In [ ]:
from pathlib import Path
from IPython.display import Video
mp4 = Path(traj.dir) / 'output.mp4'
Video(str(mp4), embed=True, width=640) if mp4.exists() else print('no video for this run')